# Task 2.2 - Visualizing Patch Attention

Use the attention weights of a pre-trained ViT to highlight which image patches drove the classification decision.

- **Model**: Hugging Face `google/vit-base-patch16-224` (ViT-B/16, 86M params, ImageNet-21k -> ImageNet-1k fine-tune)
- **Attention output**: passing `output_attentions=True` makes the model return a per-layer tuple of matrices of shape `(batch, heads, seq, seq)` with `seq = 196 patches + 1 [CLS] = 197`
- **Plan**: take the [CLS] row of the last layer, average over heads, reshape the 196 patch weights into 14 x 14, upsample to image resolution and overlay as a semi-transparent red heat map

> First run downloads the model (~330 MB) into the Hugging Face cache.

## 1. Model

In [ ]:
%pip install transformers pillow

In [ ]:
import os
import urllib.request
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(model_name).eval().to(device)
print(f"{model_name} | device: {device}")

## 2. Input images

In [ ]:
os.makedirs("images", exist_ok=True)

sources = [
    ("images/husky.jpg",
     "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
     "husky"),
    ("images/tench.jpg",
     "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n01440764_tench.JPEG",
     "tench"),
    ("images/shetland_sheepdog.jpg",
     "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n02105855_Shetland_sheepdog.JPEG",
     "Shetland sheepdog"),
]

for path, url, label in sources:
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    print(f"ready: {path}")

paths = sorted(p for p in glob("images/*")
               if os.path.splitext(p)[1].lower() in (".jpg", ".jpeg", ".png"))
print(f"{len(paths)} image(s)")

## 3. Extract the final-layer [CLS] attention

In [ ]:
cls_attn_maps = {}
for path in paths:
    inputs = processor(images=Image.open(path).convert("RGB"), return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    final_attn = outputs.attentions[-1]
    cls_attn = final_attn[:, :, 0, 1:].mean(dim=1)
    map_14 = cls_attn.reshape(14, 14).cpu().numpy()

    pred = outputs.logits.argmax(-1).item()
    conf = torch.softmax(outputs.logits, -1)[0, pred].item()
    cls_attn_maps[path] = (map_14, model.config.id2label[pred], conf)

    print(f"{os.path.basename(path):<24} final attn {tuple(final_attn.shape)} "
          f"-> CLS row over {cls_attn.shape[-1]} patches -> {tuple(map_14.shape)}")

## 4. Visualization: semi-transparent red attention overlay

In [ ]:
def attention_overlay(path, map_14):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    heat = np.array(Image.fromarray(map_14).resize((w, h), Image.Resampling.BICUBIC))
    img_np = np.array(img).astype(np.float64) / 255.0
    red = np.zeros_like(img_np)
    red[..., 0] = 1.0
    alpha = 0.6 * heat[..., None]
    return img, img_np, heat, (1.0 - alpha) * img_np + alpha * red


n = len(cls_attn_maps)
fig, axes = plt.subplots(3, n, figsize=(4.5 * n, 12))
axes = np.atleast_2d(axes)
for j, path in enumerate(paths):
    map_14, label, conf = cls_attn_maps[path]
    img, img_np, heat, merged = attention_overlay(path, map_14)

    axes[0, j].imshow(img)
    axes[0, j].set_title(f"{label} ({conf:.0%})", fontsize=9)
    axes[0, j].axis("off")

    axes[1, j].imshow(heat, cmap="jet")
    axes[1, j].set_title("attention (14x14, upsampled)", fontsize=9)
    axes[1, j].axis("off")

    axes[2, j].imshow(merged)
    axes[2, j].set_title("red overlay", fontsize=9)
    axes[2, j].axis("off")

plt.tight_layout()
plt.savefig("vit_attention.png", dpi=150)
plt.show()

## 5. Assessment

Compare with the predictions from Task 2.1. For each image:

- Does the [CLS] attention concentrate on the object (e.g. the husky's face) or does it spread over background / grid-like patterns?
- The map is averaged over all 12 heads; a single head can be sharper (or noisier) than the average - note if the mean looks diffuse.

Write one sentence per image stating which regions the model attended to and whether that matches the predicted class.